# Autodata 3.45 — analiză statică (fără Azure)

Acest notebook citește **doar fișierele de inventar și pachetele ZIP permise** din Google Drive, analizează antetele fișierelor tehnice `.IDF`, documentele XML și metadatele altor fișiere, fără să execute aplicații Windows, instalatoare, crack-uri, keygen-uri sau emulatoare.

**Cum rulezi de pe telefon:** apasă **Runtime → Run all** și autentifică-te cu contul Google Drive în care se află folderul Autodata.

**Surse:** folderul `Autodata_3_45_EXTRACTED`, ID `1XqpDaYI5zkdUXY85wLSvpsklatT0P0Bd`, care conține `INVENTAR.json`, 16 ZIP-uri și `TRANSFER_FINALIZAT.txt`.

**Ieșire:** subfolderul `AUTODATA_ANALIZA_STATICA` din același folder Drive, cu `audit_summary.json`, `audit_report.txt` și rapoarte pentru fiecare ZIP. Progresul fiecărui ZIP se salvează separat; dacă runtime-ul se oprește, notebookul poate sări peste pachetele deja analizate.

**Reguli:** dacă Google respinge descărcarea unui ZIP ca suspect, notebookul îl marchează `blocked_by_provider` și trece la următorul; **nu încearcă să ocolească protecția**. Nu presupune că formatul intern `.IDF` este cunoscut: raportează doar fapte observate (semnătură, eșantion, structură XML, dimensiuni, hashuri verificate). Fișierele executabile și instrucțiunile de activare nu sunt inspectate în conținut. Colab are limite de timp și disc; pentru pachete nereușite raportează motivul.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth, io, os, re, json, time, math, hashlib, zipfile, tempfile, shutil, collections, zlib
from pathlib import Path
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaInMemoryUpload
from googleapiclient.errors import HttpError

FOLDER_ID = "1XqpDaYI5zkdUXY85wLSvpsklatT0P0Bd"
OUTPUT_NAME = "AUTODATA_ANALIZA_STATICA"
TMP = Path("/content/autodata_readonly_audit")
TMP.mkdir(exist_ok=True)
MAX_XML_BYTES = 2_000_000
SAMPLE_BYTES = 8192
MAX_PACK_BYTES = 205_000_000

creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive = build("drive","v3",credentials=creds,cache_discovery=False)

def list_items(folder_id):
    found = []
    token = None
    while True:
        resp=drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,size,mimeType)",
            pageSize=100,pageToken=token).execute(num_retries=4)
        found.extend(resp.get("files",[]))
        token=resp.get("nextPageToken")
        if not token: return found

source_by_name = {f["name"]: f for f in list_items(FOLDER_ID)}
required = {"INVENTAR.json","TRANSFER_FINALIZAT.txt"}
if not required.issubset(source_by_name):
    raise RuntimeError("Lipsesc fișierele de confirmare/inventar din folderul Autodata.")
data = drive.files().get_media(fileId=source_by_name["INVENTAR.json"]["id"]).execute(num_retries=4)
manifest = json.loads(data.decode("utf-8-sig"))
outputs=[f for f in list_items(FOLDER_ID) if f["name"]==OUTPUT_NAME and f["mimeType"]=="application/vnd.google-apps.folder"]
out_id=outputs[0]["id"] if outputs else drive.files().create(
    body={"name":OUTPUT_NAME,"mimeType":"application/vnd.google-apps.folder","parents":[FOLDER_ID]},
    fields="id").execute(num_retries=4)["id"]
print("Sursă găsită. Fișiere:",manifest["file_count"],"pachete:",manifest["package_count"])
print("Folder rezultate: https://drive.google.com/drive/folders/"+out_id)

def upload_json(name, payload):
    content=json.dumps(payload,ensure_ascii=False,indent=2).encode("utf-8")
    existing=next((f for f in list_items(out_id) if f["name"]==name),None)
    media=MediaInMemoryUpload(content,mimetype="application/json",resumable=False)
    if existing:
        return drive.files().update(fileId=existing["id"],media_body=media,fields="id").execute(num_retries=4)
    return drive.files().create(body={"name":name,"parents":[out_id]},media_body=media,
         fields="id").execute(num_retries=4)

In [ ]:
# Validează inventarul, fără descărcarea fișierelor binare.
from pathlib import PurePosixPath
SHA_PATTERN=re.compile(r"^[0-9a-fA-F]{64}$")
RISK_EXT={".exe",".dll",".sys",".com",".scr",".bat",".cmd",".ps1",".vbs",".js",".msi",".reg",".chm"}
RISK_NAMES=("keygen","emulator","crack","license.exe","activation","activator","patcher")
TECH_EXT={".idf",".xml",".dat",".bmp"}
packages=collections.defaultdict(dict)
bad=[]
classified=collections.Counter()
sizes=collections.Counter()
all_chunks=0
for f in manifest["files"]:
    name=f["path"]
    path=PurePosixPath(name.replace("\\","/"))
    if path.is_absolute() or ".." in path.parts:
        bad.append("Cale invalidă: "+name)
    ext=Path(name).suffix.lower()
    dangerous=(ext in RISK_EXT or any(x in name.lower() for x in RISK_NAMES))
    category="excluded_executable_or_licensing" if dangerous else ("technical_data" if ext in TECH_EXT else "metadata_only")
    classified[category]+=1
    sizes[ext or "(none)"]+=1
    chunks=f.get("chunks",[])
    if sum(c.get("bytes",0) for c in chunks)!=f["size_bytes"]:
        bad.append("Mărimea fragmentelor diferă: "+name)
    for c in chunks:
        all_chunks+=1
        if not SHA_PATTERN.fullmatch(c.get("sha256","")):
            bad.append("SHA256 invalid: "+name)
        package=c["package"]
        entry=c["entry"]
        if entry in packages[package]:
            bad.append("Fragment duplicat în ZIP: "+package+"/"+entry)
        packages[package][entry]={"source":name,"size":c["bytes"],"sha256":c["sha256"],"category":category,"ext":ext}
if len(manifest["files"]) != manifest["file_count"]:
    bad.append("Număr fișiere neconcordant")
if sum(f["size_bytes"] for f in manifest["files"])!=manifest["total_extracted_bytes"]:
    bad.append("Total octeți neconcordant")
if len(packages) != manifest["package_count"]:
    bad.append("Număr pachete neconcordant")
if bad: raise RuntimeError("Inventarul e inconsistent: "+"; ".join(bad[:10]))
preflight={"archive":manifest["archive"],"original_sha256":manifest["sha256"],
          "file_count":len(manifest["files"]),"chunk_count":all_chunks,
          "packages_expected":len(packages),"categories":dict(classified),
          "extensions":dict(sizes),"verified":"manifest_internal_consistency_only",
          "caution":"Nu au fost citite încă fișierele binare din ZIP."}
upload_json("preflight.json",preflight)
print("PREFLIGHT=PASS",preflight)

In [ ]:
# Citește ZIP-urile permise de Google, fără unrar, fără lansarea vreunui program.
# Raport per pachet salvat în Drive; pachetele blocate de Google sunt doar raportate.
import xml.etree.ElementTree as ET
def sample_info(data):
    if not data: return {"sample_bytes":0}
    freq=collections.Counter(data)
    entropy=-sum((n/len(data))*math.log2(n/len(data)) for n in freq.values())
    sig=" ".join(f"{b:02X}" for b in data[:16])
    strings=re.findall(rb"[\x20-\x7e]{5,}",data[:SAMPLE_BYTES])
    return {"sample_bytes":len(data),"hex_header":sig,
            "sample_entropy_bits_per_byte":round(entropy,3),
            "ascii_strings_sample":[s.decode("ascii",errors="ignore")[:80] for s in strings[:12]]}

def analyze_package(package, fileid, expected_entries):
    target=TMP/package
    result={"package":package,"status":"not_started","inspected_entries":[],"errors":[],
            "skipped_active_content":[],"observed_xml":[],"idfs":[]}
    try:
        if target.exists(): target.unlink()
        with target.open("wb") as dst:
            request=drive.files().get_media(fileId=fileid)
            downloader=MediaIoBaseDownload(dst,request,chunksize=8*1024*1024)
            done=False
            while not done:
                _,done=downloader.next_chunk(num_retries=5)
        if target.stat().st_size>MAX_PACK_BYTES:
            raise RuntimeError("ZIP depășește dimensiunea permisă")
        with zipfile.ZipFile(target) as z:
            members={i.filename:i for i in z.infolist() if not i.is_dir()}
            for entry, expected in expected_entries.items():
                if entry not in members:
                    result["errors"].append({"entry":entry,"error":"missing_from_zip"})
                    continue
                info=members[entry]
                if info.file_size!=expected["size"]:
                    result["errors"].append({"entry":entry,"error":"size_mismatch"})
                    continue
                if expected["category"]=="excluded_executable_or_licensing":
                    result["skipped_active_content"].append(entry)
                    continue
                h=hashlib.sha256()
                prefix=bytearray()
                xml_data=bytearray()
                with z.open(info) as stream:
                    while True:
                        block=stream.read(1024*1024)
                        if not block: break
                        h.update(block)
                        if len(prefix)<SAMPLE_BYTES:
                            prefix.extend(block[:SAMPLE_BYTES-len(prefix)])
                        if expected["ext"]==".xml" and len(xml_data)<MAX_XML_BYTES:
                            xml_data.extend(block[:MAX_XML_BYTES-len(xml_data)])
                good=(h.hexdigest().lower()==expected["sha256"].lower())
                row={"entry":entry,"source":expected["source"],"bytes":info.file_size,
                     "sha256_verified":good}
                if not good:
                    result["errors"].append({"entry":entry,"error":"sha256_mismatch"})
                elif expected["ext"]==".idf":
                    result["idfs"].append({**row,**sample_info(prefix)})
                elif expected["ext"]==".xml" and info.file_size <= MAX_XML_BYTES:
                    try:
                        root=ET.fromstring(xml_data)
                        result["observed_xml"].append({**row,"root_tag":root.tag,
                            "root_attributes":list(root.attrib.keys())[:20],
                            "children_first_level":[x.tag for x in list(root)[:20]]})
                    except ET.ParseError as err:
                        result["observed_xml"].append({**row,"parse_error":str(err)})
                result["inspected_entries"].append(row)
            result["status"]="analyzed_with_errors" if result["errors"] else "analyzed"
    except HttpError as e:
        raw=e.content.decode("utf-8","replace") if isinstance(e.content,bytes) else str(e.content)
        if e.resp.status==403 and any(s in raw for s in ("cannotDownloadAbusiveFile","malware","spam")):
            result["status"]="blocked_by_provider"
            result["errors"].append("Google Drive a blocat descărcarea pentru potențial malware/spam; nu este ocolită protecția.")
        else:
            result["status"]="api_error"
            result["errors"].append(f"HTTP {e.resp.status}: "+raw[:500])
    except Exception as err:
        result["status"]="error"
        result["errors"].append(type(err).__name__+": "+str(err)[:500])
    finally:
        if target.exists(): target.unlink()
    return result

all_results={}
saved_by_name={f["name"]:f for f in list_items(out_id)}
for index, (pkg, expected_entries) in enumerate(sorted(packages.items()),1):
    checkpoint="audit_"+pkg+".json"
    if checkpoint in saved_by_name:
        # Checkpoints previously completed are not fetched/reprocessed. They will
        # be present in Google Drive even after Colab resets.
        print(f"{index}/{len(packages)} existent: {checkpoint}")
        continue
    if pkg not in source_by_name:
        result={"package":pkg,"status":"missing_package","errors":["ZIP inexistent în Drive"]}
    else:
        result=analyze_package(pkg,source_by_name[pkg]["id"],expected_entries)
    upload_json(checkpoint,result)
    all_results[pkg]=result
    print(f"{index}/{len(packages)}",pkg,result["status"],
          "data:",len(result.get("inspected_entries",[])),
          "IDF:",len(result.get("idfs",[])))
print("SCAN_STAGE_COMPLETE — toate rezultatele sunt în folderul Drive.")

In [ ]:
# Agregă toate rapoartele per ZIP din Drive, produce concluzii verificate.
output_list={f["name"]:f for f in list_items(out_id)}
per_package=[]
for pkg in sorted(packages):
    key="audit_"+pkg+".json"
    if key not in output_list:
        per_package.append({"package":pkg,"status":"missing_report"})
        continue
    blob=drive.files().get_media(fileId=output_list[key]["id"]).execute(num_retries=4)
    per_package.append(json.loads(blob.decode("utf-8")))
counts=collections.Counter(r.get("status") for r in per_package)
verified=sum(len([x for x in r.get("inspected_entries",[]) if x.get("sha256_verified")]) for r in per_package)
idfs=[x for r in per_package for x in r.get("idfs",[])]
xmls=[x for r in per_package for x in r.get("observed_xml",[])]
errors=[{"package":r.get("package"),"problems":r.get("errors")} for r in per_package if r.get("errors")]
summary={"source_folder_id":FOLDER_ID,"file_count_expected":manifest["file_count"],
         "total_chunks_expected":all_chunks,"packages":dict(counts),
         "content_chunks_sha256_verified":verified,"idf_observations":idfs,
         "xml_observations":xmls,"problems":errors,
         "disclaimer":"Datele .IDF sunt analizate doar la nivel de antet/format. Nu s-a confirmat structura internă.",
         "executables_ran":False,"security_blocks_bypassed":False}
upload_json("audit_summary.json",summary)
lines=["AUTODATA 3.45 / RAPORT ANALIZA STATICA", "",
       f"Pachete: {dict(counts)}",
       f"Fragmente non-executabile verificate SHA256: {verified}",
       f"Antete .IDF observate: {len(idfs)}",
       f"XML inspectate: {len(xmls)}",
       f"Erori/blocaje raportate: {len(errors)}", "",
       "Nu a fost executat niciun fișier al aplicației; formatul .IDF nu a fost dedus fără dovezi.",
       "Detaliile se află în audit_summary.json și în audit_autodata_part_XXXX.zip.json."]
content="\n".join(lines).encode("utf-8")
exists=next((f for f in list_items(out_id) if f["name"]=="audit_report.txt"),None)
media=MediaInMemoryUpload(content,mimetype="text/plain")
if exists: drive.files().update(fileId=exists["id"],media_body=media).execute(num_retries=4)
else: drive.files().create(body={"name":"audit_report.txt","parents":[out_id]},media_body=media).execute(num_retries=4)
print("\n".join(lines))
print("ANALYSIS_REPORTS_SAVED=PASS")
print("https://drive.google.com/drive/folders/"+out_id)